# 蝶威量化因子挑战赛 —— 最优策略方案

---

## 一、比赛内容简介（通俗版）

这是一个**量化投资因子挖掘比赛**，由上海蝶威私募基金管理有限公司（DeepWin）举办。

**简单来说，你需要做的事情是：**

1. 你拿到的是 A 股沪深300成分股在 2023~2024 年间的**高频快照数据**（类似于每隔几秒钟拍一张股价、挂单的"照片"）。
2. 你的任务是从这些数据中**发明一个"评分公式"（即因子）**，每隔15分钟给每只股票打一个分。
3. 评分越高的股票，应该在接下来的表现越好（涨得多）。系统会自动用你的评分去选股，每天选评分最高的60只股票买入并持有。
4. 评分的好坏通过回测来衡量：看你选的股票组合能跑赢沪深300指数多少，夏普比率（风险调整后收益）是多少。
5. 最终得分 = 0.3 × 单因子分析排名 + 0.7 × 单因子回测排名。

**关键约束：**
- 代码必须在平台 Notebook 上运行，CPU 限时 9 小时
- 不能访问外网，不能用外部数据
- 输出格式必须是 `date | instrument | factor` 三列

---

## 二、官方 Demo 逻辑解析

官方 Demo (`demo/demo.ipynb`) 构建了一个**加权订单簿压力因子（Weighted Order Book Pressure）**：

1. **数据预处理**：将快照时间按 15 分钟切分为窗口（09:30-09:45, 09:45-10:00, ...）
2. **中间价**：`mid_price = (ask_price1 + bid_price1) / 2`
3. **加权买/卖量**：对 1~5 档挂单量做指数衰减加权 `W = V1×1.0 + V2×e^{-0.3} + V3×e^{-0.6} + ...`
4. **订单簿不平衡度**：`Imbalance = (W_bid - W_ask) / (W_bid + W_ask)`
5. **价差因子**：`Spread = (ask1 - bid1) / mid_price`
6. **原始压力**：`Pressure = (Imbalance / √|Spread|) × Imbalance`
7. **窗口聚合**：Z-Score 标准化压力值
8. **波动率调整**：高波动（>80%分位数）时衰减至 0.7 倍
9. **输出**：`factor = tanh(adjusted_pressure) × (-1)`（取反表示卖压大 → 做空）

**优点**：利用了盘口深度数据，考虑了波动率调整。
**缺点**：仅使用订单簿信息，忽略了价格动量和量价关系等正交信号。

---

## 三、现有策略分析

### 1. 价格排序因子（价格排序.ipynb）
- **逻辑**：低价股因子值高（接近1），高价股因子值低（接近0）
- **优点**：简单直观，低价效应在 A 股有一定历史有效性
- **缺点**：信号过于简单，容易被 BARRA 风格因子中和；低价股可能有质量风险

### 2. 动量反转因子（动量反转.ipynb / momentum_py-v3.ipynb）
- **逻辑**：`factor = -tanh((close - avg) / avg × 10)`，15分钟内收盘价高于均价则做空
- **优点**：捕捉日内均值回归，逻辑清晰
- **缺点**：单一维度，高波动时均值回归假设可能失效

### 3. 订单簿压力因子（订单薄压力.ipynb）
- **逻辑**：与 Demo 几乎相同，但因子方向为正（factor × 1）
- **优点**：利用微观结构信息，含波动率调整
- **缺点**：与 Demo 高度重合，缺乏差异化

### 4. 量价相关性因子（量价相关性.ipynb）
- **逻辑**：计算15分钟窗口内价格收益率与成交量增量的皮尔逊相关系数
- **优点**：捕捉量价背离/同步，与其他因子正交性好
- **缺点**：窗口内样本可能不足（<3），相关系数不稳定

### 5. 价格&反转复合因子（价格&反转复合.ipynb）
- **逻辑**：`factor = 0.2 × 价格排序 + 0.8 × 动量反转`
- **优点**：多信号融合优于单因子
- **缺点**：仅融合两个弱因子，缺少微观结构维度；权重固定未优化

---

## 四、最优策略设计

### 设计思路

融合**四个正交维度**的子因子，最大化因子的信息比率：

| 维度 | 子因子 | 权重 | 信号来源 |
|------|--------|------|----------|
| 微观结构 | 加权订单簿不平衡度 | 0.35 | 5档盘口买卖力量对比 |
| 价格动量 | 日内均值回归 | 0.30 | 15min收盘价 vs 均价偏离 |
| 量价关系 | 量价相关性 | 0.20 | 价格变化与成交量增量的同步性 |
| 流动性 | 成交量加速度 | 0.15 | 成交量变化率 |

### 关键改进
1. **多因子融合**：四个子因子来自不同信息源，相关性低，组合后 IC 更稳定
2. **统一标准化**：每个子因子在截面上做 Z-Score 后再加权，避免量纲差异
3. **波动率自适应**：高波动时衰减因子强度，减少噪音交易
4. **tanh 压缩**：最终输出经 tanh 映射到 (-1, 1)，抑制极端值
5. **纯 SQL 实现**：在 DAI 数据引擎中直接计算，速度快、内存低


In [ ]:
def main(datasource, start_date, end_date):
    """
    多因子复合策略：融合订单簿不平衡、均值回归、量价相关性、成交量加速度

    四个子因子:
    1) 加权订单簿不平衡度 (Weighted Order Book Imbalance)
    2) 日内均值回归动量 (Intraday Mean Reversion)
    3) 量价相关性 (Price-Volume Correlation)
    4) 成交量加速度 (Volume Acceleration)

    每个子因子在截面上 Z-Score 标准化后加权合成，
    高波动环境下衰减因子强度，最终 tanh 压缩输出。

    Args:
        datasource (str): Datasource table name
        start_date (str): Start date in 'YYYY-MM-DD HH:MM:SS' format
        end_date (str): End date in 'YYYY-MM-DD HH:MM:SS' format

    Returns:
        pd.DataFrame: Factor data with columns ['date', 'instrument', 'factor']
    """
    import pandas as pd
    import dai

    # 子因子权重（可调优）
    w_imbalance = 0.35
    w_reversal = 0.30
    w_pv_corr = 0.20
    w_vol_accel = 0.15

    sql = f"""
    -- 优化设置
    SET preserve_insertion_order=false;
    SET threads=4;

    -- ============================================================
    -- Step 1: 基础快照数据 + 15分钟时间窗口划分
    -- ============================================================
    WITH cte_snapshot AS (
        SELECT
            date,
            instrument_id,
            price,
            volume,

            strftime(date, '%Y-%m-%d') AS trading_day,

            -- 中间价
            (ask_price1 + bid_price1) / 2.0 AS mid_price,

            -- 加权5档买方量（指数衰减）
            (
                COALESCE(bid_volume1, 0) * 1.0 +
                COALESCE(bid_volume2, 0) * EXP(-0.3) +
                COALESCE(bid_volume3, 0) * EXP(-0.6) +
                COALESCE(bid_volume4, 0) * EXP(-0.9) +
                COALESCE(bid_volume5, 0) * EXP(-1.2)
            ) AS weight_bid,

            -- 加权5档卖方量（指数衰减）
            (
                COALESCE(ask_volume1, 0) * 1.0 +
                COALESCE(ask_volume2, 0) * EXP(-0.3) +
                COALESCE(ask_volume3, 0) * EXP(-0.6) +
                COALESCE(ask_volume4, 0) * EXP(-0.9) +
                COALESCE(ask_volume5, 0) * EXP(-1.2)
            ) AS weight_ask,

            -- 加权订单簿不平衡度
            (weight_bid - weight_ask) / (weight_bid + weight_ask + 1e-8) AS weighted_imbalance,


            -- 15分钟时间窗口
            CASE
                WHEN strftime(date, '%H%M') >= '0930' AND strftime(date, '%H%M') < '0945' THEN 94500
                WHEN strftime(date, '%H%M') >= '0945' AND strftime(date, '%H%M') < '1000' THEN 100000
                WHEN strftime(date, '%H%M') >= '1000' AND strftime(date, '%H%M') < '1015' THEN 101500
                WHEN strftime(date, '%H%M') >= '1015' AND strftime(date, '%H%M') < '1030' THEN 103000
                WHEN strftime(date, '%H%M') >= '1030' AND strftime(date, '%H%M') < '1045' THEN 104500
                WHEN strftime(date, '%H%M') >= '1045' AND strftime(date, '%H%M') < '1100' THEN 110000
                WHEN strftime(date, '%H%M') >= '1100' AND strftime(date, '%H%M') < '1115' THEN 111500
                WHEN strftime(date, '%H%M') >= '1115' AND strftime(date, '%H%M') <= '1130' THEN 113000
                WHEN strftime(date, '%H%M') >= '1300' AND strftime(date, '%H%M') < '1315' THEN 131500
                WHEN strftime(date, '%H%M') >= '1315' AND strftime(date, '%H%M') < '1330' THEN 133000
                WHEN strftime(date, '%H%M') >= '1330' AND strftime(date, '%H%M') < '1345' THEN 134500
                WHEN strftime(date, '%H%M') >= '1345' AND strftime(date, '%H%M') < '1400' THEN 140000
                WHEN strftime(date, '%H%M') >= '1400' AND strftime(date, '%H%M') < '1415' THEN 141500
                WHEN strftime(date, '%H%M') >= '1415' AND strftime(date, '%H%M') < '1430' THEN 143000
                WHEN strftime(date, '%H%M') >= '1430' AND strftime(date, '%H%M') < '1445' THEN 144500
                WHEN strftime(date, '%H%M') >= '1445' AND strftime(date, '%H%M') < '1457' THEN 150000
                ELSE -1
            END AS time_segment

        FROM {datasource}
        WHERE time_segment != -1
          AND mid_price IS NOT NULL
          AND mid_price > 0
    ),

    -- ============================================================
    -- Step 2: 逐笔差分（用于量价相关性和成交量加速度）
    -- ============================================================
    cte_delta AS (
        SELECT
            *,
            LAG(mid_price) OVER (
                PARTITION BY instrument_id, trading_day, time_segment
                ORDER BY date
            ) AS prev_mid_price,
            LAG(volume) OVER (
                PARTITION BY instrument_id, trading_day, time_segment
                ORDER BY date
            ) AS prev_volume,

            -- 收益率
            (mid_price - prev_mid_price) / (prev_mid_price + 1e-8) AS ret,
            -- 成交量增量
            (volume - prev_volume) AS vol_delta
        FROM cte_snapshot
    ),

    -- ============================================================
    -- Step 3: 15分钟窗口聚合 - 计算四个子因子的原始值
    -- ============================================================
    cte_window AS (
        SELECT
            instrument_id,
            trading_day,
            time_segment,

            -- 子因子1: 加权订单簿不平衡度（窗口均值）
            avg(weighted_imbalance) AS f1_imbalance,

            -- 子因子2: 日内均值回归
            argMax(mid_price, date) AS close_mid,
            avg(mid_price) AS avg_mid,
            -1.0 * tanh(((close_mid - avg_mid) / (avg_mid + 1e-8)) * 10.0) AS f2_reversal,

            -- 子因子3: 量价相关性
            CASE
                WHEN SUM(CASE WHEN prev_mid_price IS NOT NULL AND prev_volume IS NOT NULL THEN 1 ELSE 0 END) < 3
                THEN 0
                ELSE COALESCE(CORR(ret, vol_delta), 0)
            END AS f3_pv_corr,

            -- 子因子4: 成交量加速度（窗口结束 vs 窗口开始的成交量变化率）
            argMax(volume, date) AS end_volume,
            argMin(volume, date) AS start_volume,
            avg(volume) AS avg_volume,
            CASE
                WHEN avg_volume > 0
                THEN (end_volume - start_volume) / (avg_volume + 1e-8)
                ELSE 0
            END AS f4_vol_accel,

            -- 波动率（用于后续调整）
            nanstd(ret) AS volatility,
            COUNT(*) AS n_obs

        FROM cte_delta
        GROUP BY instrument_id, trading_day, time_segment
    ),

    -- ============================================================
    -- Step 4: 截面Z-Score标准化（每个 trading_day + time_segment 截面内）
    -- ============================================================
    cte_zscore AS (
        SELECT
            instrument_id,
            trading_day,
            time_segment,
            volatility,

            -- Z-Score: (x - mean) / std
            (f1_imbalance - AVG(f1_imbalance) OVER w_cs)
                / (NULLIF(STDDEV(f1_imbalance) OVER w_cs, 0) + 1e-8) AS z_imbalance,

            (f2_reversal - AVG(f2_reversal) OVER w_cs)
                / (NULLIF(STDDEV(f2_reversal) OVER w_cs, 0) + 1e-8) AS z_reversal,

            (f3_pv_corr - AVG(f3_pv_corr) OVER w_cs)
                / (NULLIF(STDDEV(f3_pv_corr) OVER w_cs, 0) + 1e-8) AS z_pv_corr,

            (f4_vol_accel - AVG(f4_vol_accel) OVER w_cs)
                / (NULLIF(STDDEV(f4_vol_accel) OVER w_cs, 0) + 1e-8) AS z_vol_accel,

            -- 波动率分位数阈值
            PERCENT_RANK() OVER (
                PARTITION BY trading_day, time_segment
                ORDER BY volatility
            ) AS vol_pct_rank

        FROM cte_window
        WINDOW w_cs AS (PARTITION BY trading_day, time_segment)
    ),

    -- ============================================================
    -- Step 5: 加权合成 + 波动率调整 + tanh压缩
    -- ============================================================
    cte_composite AS (
        SELECT
            instrument_id,
            trading_day,
            time_segment,

            -- 原始复合因子
            (
                {w_imbalance} * COALESCE(z_imbalance, 0) +
                {w_reversal}  * COALESCE(z_reversal, 0) +
                {w_pv_corr}   * COALESCE(z_pv_corr, 0) +
                {w_vol_accel} * COALESCE(z_vol_accel, 0)
            ) AS raw_composite,

            -- 波动率调整：高波动（>80%分位）时衰减至0.7倍
            CASE
                WHEN vol_pct_rank > 0.8
                THEN raw_composite * 0.7
                ELSE raw_composite
            END AS adj_composite,

            -- tanh压缩到(-1, 1)
            tanh(adj_composite) AS factor

        FROM cte_zscore
    )

    -- ============================================================
    -- Step 6: 输出标准格式
    -- ============================================================
    SELECT
        CAST(CONCAT(
            c.trading_day, ' ',
            strftime(strptime(LPAD(c.time_segment, 6, '0'), '%H%M%S'), '%H:%M:%S')
        ) AS DATETIME) AS date,
        all_instruments.instrument,
        c.factor
    FROM cte_composite c
    LEFT JOIN all_instruments USING (instrument_id)
    """

    df = dai.query(sql, filters={'date': [start_date, end_date]}).df()
    return df


if __name__ == '__main__':
    """
    开发调试专用模块：分块循环回测引擎
    """
    from bigmodule import M
    import pandas as pd
    import structlog
    import gc

    logger = structlog.get_logger()
    datasource = 'cpt_dwc_2026_stock_hs300_snapshot'

    full_start_date = '2023-01-01'
    full_end_date = '2024-12-01'

    date_ranges = pd.date_range(start=full_start_date, end=full_end_date, freq='MS')
    all_results = []

    logger.info(f"Starting Multi-Factor Composite Backtest: {full_start_date} to {full_end_date}")

    for start_dt in date_ranges:
        current_start = start_dt.strftime('%Y-%m-%d 00:00:00')
        current_end = (start_dt + pd.offsets.MonthEnd(0)).strftime('%Y-%m-%d 23:59:59')
        logger.info(f"Processing Chunk: {current_start} => {current_end}")

        try:
            df_chunk = main(datasource, current_start, current_end)

            if df_chunk is not None and not df_chunk.empty:
                all_results.append(df_chunk)
                logger.info(f"Chunk Done. Rows: {len(df_chunk)}")
            else:
                logger.warning(f"Chunk Empty: {current_start}")

            del df_chunk
            gc.collect()

        except Exception as e:
            logger.error(f"Error in chunk {current_start}: {e}")

    if all_results:
        logger.info("Concatenating all chunks...")
        final_data = pd.concat(all_results, ignore_index=True)
        final_data.sort_values(by=['date', 'instrument'], inplace=True)

        logger.info(f"All Done! Final Shape: {final_data.shape}")
        logger.info(f"Sample:\n{final_data.head()}")

        logger.info("Starting Evaluation...")
        try:
            results = M.eval_dwc._latest(data=final_data)
        except Exception as e:
            logger.warning(f"Evaluation failed (local env might miss modules): {e}")
            print("Data preview:", final_data.head())
    else:
        logger.error("No data generated.")
